In [1]:
household_ids = ["A01","A02","A03","A04","A05","A06","A07","A08"]
income_text = ["1200", "0", "1500", "700", "2400", "800", "0", "1800"]
consumptions = [1250, 150, 1600, 850, 2600, 950, 0, 1400]
household_sizes = [2, 1, 3, 1, 4, 2, 2, 3]

# 1. Trace values and notebook state

## Prediction before running the file

## (a) [1200,800,0,600]; 600 4; 20%

For the first value:
both incomes and saved_incomes are pointing at the same list, therefore changing the same list.
1. incomes[0] changes first value to 1200
2. saved_incomes.append(600) adds 600 at the end of incomes list (which saved_income also points at)

For the second value:
saved_incomes[-1] is the last value of the saved_incomes list, which is 600
len(incomes) is the length of the list, which is 4 

Third value:
first_share = (incomes[0] - 800) / incomes[0] = (1000 - 800) / 1000 = 200 / 1000 = 0.2
Fstring ":.0%" makes it a percent with no digits after comma,which is 20%

In [2]:
incomes = [1000,800,0]
saved_incomes = incomes
first_share = (incomes[0] - 800) / incomes[0]
incomes[0] = 1200
saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0, 600]
600 4
20%


## (b) `saved_incomes = incomes` gives both names references to the same mutable list, so appending through either name changes that shared list. `first_share` is a separate float calculated once using the original first income. Changing `incomes[0]` later does not recalculate the expression or update the stored float.

# (c)

In [3]:
#(c)

incomes = [1000,800,0]
saved_incomes = incomes.copy()
incomes[0] = 1200
first_share = (incomes[0] - 800) / incomes[0]

saved_incomes.append(600)
print(incomes)
print(saved_incomes[-1], len(incomes))
print(f"{first_share:.0%}")

[1200, 800, 0]
600 3
33%


# 2. Implement the policy as functions

In [4]:
# (a)

def saving_share(income, consumption):
    if income < 0 or consumption < 0:
        raise ValueError("Income and consumption must be non-negative.")
    if income == 0:
        return None
    else:
        return (income - consumption) / income

# (b)

def transfer(income, consumption, size):

    if income < 0 or consumption < 0 or size <= 0:
        raise ValueError("Income and consumption must be non-negative, and size must be positive.")
    
    if income / size > 600:
        return 0
    else:
        return min(100*size, max(consumption - income, 0))

# (c)

def post_policy_share(income, consumption, size):
    new_income = income + transfer(income, consumption, size)
    return saving_share(new_income, consumption)


""" Design explanation:
Parameters are what is used to calculate in functions, so income, consumption and size.
Arguments are the actual values being passed to the function when it is called.

Return is needed so other functions can use the values being outputted by the function.

Checks for division are used to avoid divison by zero.

"""

' Design explanation:\nParameters are what is used to calculate in functions, so income, consumption and size.\nArguments are the actual values being passed to the function when it is called.\n\nReturn is needed so other functions can use the values being outputted by the function.\n\nChecks for division are used to avoid divison by zero.\n\n'

# 3. Build and interpret a checked report

In [5]:
# (a)
same_length = (
    len(household_ids)
    == len(income_text)
    == len(consumptions)
    == len(household_sizes)
)
if not same_length:
    raise ValueError("All input lists must have the same length.")

# Convert income text to numbers.
income_numbers = []
for text in income_text:
    income_numbers.append(int(text))

transfer_amounts = []
saving_amounts_before = []
saving_amounts_after = []
incomes_after_support = []
saving_shares_before = []
saving_shares_after = []

# (b)

for i in range(len(household_ids)):
    household_id = household_ids[i]
    income = income_numbers[i]
    consumption_amount = consumptions[i]
    size = household_sizes[i]

    transfer_amount = transfer(income, consumption_amount, size)
    saving_before = income - consumption_amount
    saving_after = income + transfer_amount - consumption_amount
    share_before = saving_share(income, consumption_amount)
    share_after = post_policy_share(income, consumption_amount, size)

    transfer_amounts.append(transfer_amount)
    saving_amounts_before.append(saving_before)
    saving_amounts_after.append(saving_after)
    incomes_after_support.append(income + transfer_amount)
    saving_shares_before.append(share_before)
    saving_shares_after.append(share_after)

    if share_before is None:
        share_before_display = "undefined"
    else:
        share_before_display = f"{share_before:.2%}"

    if share_after is None:
        share_after_display = "undefined"
    else:
        share_after_display = f"{share_after:.2%}"

    print(f"Household {household_id}: Original income: €{income:.2f}, Consumption: €{consumption_amount:.2f}, Size: {size}, Transfer: €{transfer_amount:.2f}, Saving before: €{saving_before:.2f}, Saving after: €{saving_after:.2f}, Share before: {share_before_display}, Share after: {share_after_display}")

Household A01: Original income: €1200.00, Consumption: €1250.00, Size: 2, Transfer: €50.00, Saving before: €-50.00, Saving after: €0.00, Share before: -4.17%, Share after: 0.00%
Household A02: Original income: €0.00, Consumption: €150.00, Size: 1, Transfer: €100.00, Saving before: €-150.00, Saving after: €-50.00, Share before: undefined, Share after: -50.00%
Household A03: Original income: €1500.00, Consumption: €1600.00, Size: 3, Transfer: €100.00, Saving before: €-100.00, Saving after: €0.00, Share before: -6.67%, Share after: 0.00%
Household A04: Original income: €700.00, Consumption: €850.00, Size: 1, Transfer: €0.00, Saving before: €-150.00, Saving after: €-150.00, Share before: -21.43%, Share after: -21.43%
Household A05: Original income: €2400.00, Consumption: €2600.00, Size: 4, Transfer: €200.00, Saving before: €-200.00, Saving after: €0.00, Share before: -8.33%, Share after: 0.00%
Household A06: Original income: €800.00, Consumption: €950.00, Size: 2, Transfer: €150.00, Saving

## (c) Compare totals and averages


In [6]:
total_transfer = 0
positive_transfer_households = 0
negative_saving_before_households = 0
negative_saving_after_households = 0

for i in range(len(household_ids)):
    transfer_amount = transfer_amounts[i]
    saving_before = saving_amounts_before[i]
    saving_after = saving_amounts_after[i]

    total_transfer += transfer_amount

    if transfer_amount > 0:
        positive_transfer_households += 1
    if saving_before < 0:
        negative_saving_before_households += 1
    if saving_after < 0:
        negative_saving_after_households += 1

print(f"Total transfer: €{total_transfer:.2f}")
print(f"Households receiving a positive transfer: {positive_transfer_households} of {len(household_ids)}")
print(f"Households with negative saving before: {negative_saving_before_households} of {len(household_ids)}")
print(f"Households with negative saving after: {negative_saving_after_households} of {len(household_ids)}")

Total transfer: €600.00
Households receiving a positive transfer: 5 of 8
Households with negative saving before: 6 of 8
Households with negative saving after: 2 of 8


In [7]:
# (c) Pooled shares and arithmetic means

pooled_saving_share_before = sum(saving_amounts_before) / sum(income_numbers)
pooled_saving_share_after = sum(saving_amounts_after) / sum(incomes_after_support)

defined_saving_shares_before = []
defined_saving_shares_after = []
for share in saving_shares_before:
    if share is not None:
        defined_saving_shares_before.append(share)
for share in saving_shares_after:
    if share is not None:
        defined_saving_shares_after.append(share)

mean_defined_saving_share_before = sum(defined_saving_shares_before) / len(defined_saving_shares_before)
mean_defined_saving_share_after = sum(defined_saving_shares_after) / len(defined_saving_shares_after)

# Restrict both periods to households whose share was defined before support.
saving_shares_before_same_households = []
saving_shares_after_same_households = []
for i in range(len(household_ids)):
    if saving_shares_before[i] is not None:
        saving_shares_before_same_households.append(saving_shares_before[i])
        saving_shares_after_same_households.append(saving_shares_after[i])

mean_saving_share_before_same_households = sum(saving_shares_before_same_households) / len(saving_shares_before_same_households)
mean_saving_share_after_same_households = sum(saving_shares_after_same_households) / len(saving_shares_after_same_households)

print(f"Pooled saving share before: {pooled_saving_share_before:.2%} (sum saving €{sum(saving_amounts_before):.2f} / sum income €{sum(income_numbers):.2f})")
print(f"Pooled saving share after: {pooled_saving_share_after:.2%} (sum saving €{sum(saving_amounts_after):.2f} / sum income after €{sum(incomes_after_support):.2f})")
print(f"Mean of defined household saving shares before: {mean_defined_saving_share_before:.2%} (n = {len(defined_saving_shares_before)})")
print(f"Mean of defined household saving shares after: {mean_defined_saving_share_after:.2%} (n = {len(defined_saving_shares_after)})")
print(f"Mean after support, restricted to households defined before: {mean_saving_share_after_same_households:.2%} (n = {len(saving_shares_after_same_households)}; before = {mean_saving_share_before_same_households:.2%})")


Pooled saving share before: -4.76% (sum saving €-400.00 / sum income €8400.00)
Pooled saving share after: 2.22% (sum saving €200.00 / sum income after €9000.00)
Mean of defined household saving shares before: -6.19% (n = 6)
Mean of defined household saving shares after: -7.03% (n = 7)
Mean after support, restricted to households defined before: 0.13% (n = 6; before = -6.19%)


Pooled saving share rises from -4.76% to 2.22%: total saving moves from -€400 to €200, while total income rises from €8,400 to €9,000 as transfer_amounts are added. The mean of defined household shares falls from -6.19% over 6 households to -7.03% over 7, which seems to contradict that improvement. The reason is A02: with zero income its share was undefined, but after a €100 transfer its income is €100 and its share is -50%, so it enters the mean and pulls it down. Holding the population fixed at the 6 households defined before support gives +0.13% after.


## (d) Use an inline function


In [8]:
# (d) Sort defined pre-policy shares by distance from a target of 0.10

target_saving_share = 0.10
defined_pre_policy_saving_shares = sorted(
    defined_saving_shares_before,
    key=lambda share_value: abs(share_value - target_saving_share),
)

print("Defined pre-policy shares, nearest to 0.10 first:")
print(defined_pre_policy_saving_shares)


Defined pre-policy shares, nearest to 0.10 first:
[0.2222222222222222, -0.041666666666666664, -0.06666666666666667, -0.08333333333333333, -0.1875, -0.21428571428571427]


The lambda receives one element of `defined_saving_shares_before`, that is, a single defined pre-policy saving share, and returns its absolute distance from the target 0.10. `sorted` uses these returned distances as keys, so the share nearest the target comes first. The sorted list still holds the original signed shares, not the distances: A08's +22.22% is nearest, then A01's -4.17%, while A04's -21.43% is last.


## (e) Test a counterfactual


In [9]:
# (e) Counterfactual: raise A01's income from 1200 to 1202

original_income_numbers = income_numbers.copy()
counterfactual_income_numbers = income_numbers.copy()
counterfactual_income_numbers[0] = 1202

a01_consumption = consumptions[0]
a01_size = household_sizes[0]

a01_transfer_baseline = transfer(original_income_numbers[0], a01_consumption, a01_size)
a01_transfer_counterfactual = transfer(counterfactual_income_numbers[0], a01_consumption, a01_size)

a01_income_after_support_baseline = original_income_numbers[0] + a01_transfer_baseline
a01_income_after_support_counterfactual = counterfactual_income_numbers[0] + a01_transfer_counterfactual

a01_saving_after_support_baseline = a01_income_after_support_baseline - a01_consumption
a01_saving_after_support_counterfactual = a01_income_after_support_counterfactual - a01_consumption

print(f"A01 transfer: €{a01_transfer_baseline:.2f} -> €{a01_transfer_counterfactual:.2f} (change €{a01_transfer_counterfactual - a01_transfer_baseline:+.2f})")
print(f"A01 income after support: €{a01_income_after_support_baseline:.2f} -> €{a01_income_after_support_counterfactual:.2f} (change €{a01_income_after_support_counterfactual - a01_income_after_support_baseline:+.2f})")
print(f"A01 saving after support: €{a01_saving_after_support_baseline:.2f} -> €{a01_saving_after_support_counterfactual:.2f} (change €{a01_saving_after_support_counterfactual - a01_saving_after_support_baseline:+.2f})")

if income_numbers != original_income_numbers:
    raise ValueError("The original income list was modified.")
print(f"Original A01 income is still €{income_numbers[0]:.2f}; the input list is unchanged.")

A01 transfer: €50.00 -> €0.00 (change €-50.00)
A01 income after support: €1250.00 -> €1202.00 (change €-48.00)
A01 saving after support: €0.00 -> €-48.00 (change €-48.00)
Original A01 income is still €1200.00; the input list is unchanged.


Raising A01's original income by €2 lowers income after support by €48. The extra €2 takes income per person from €600 to €601, so A01 stops being eligible and the €50 transfer becomes €0. Higher original income therefore does not necessarily mean higher income after support: the policy has a cap at €600 per person, and a household just above it can end up worse off than one just below.


# 4. Audit an agent-written function


In [10]:
# The agent's proposed function, kept for reference.
# The failing calls and their observed errors are in the Markdown cell below.

def proposed_mean(incomes, consumptions):
    total = 0
    count = 0
    for income, consumption in zip(incomes, consumptions):
        share = saving_share(income, consumption)
        if share is not None and share != 0:
            total = 0
            total = total + share
            count = count + 1
    return total / count


## (a) Four distinct defects

1. **Zero shares are dropped.** The condition `share != 0` excludes valid zero shares, although the contract says they must be included.
2. **The running total is reset.** `total = 0` inside the loop throws away every earlier share, so the numerator is only the last included share while `count` still counts all included shares.
3. **No length check.** `zip` stops at the shorter list and silently drops households. The contract requires `ValueError` for unequal lengths.
4. **No empty guard.** When `count == 0` (empty inputs or no defined shares), `total / count` raises `ZeroDivisionError`; the contract requires `None`.

The proposal does not modify its inputs, so that part of the contract is satisfied.


## (b) Repaired function


In [11]:
# (b) Repaired mean of defined saving shares

def mean_defined_share(incomes, consumptions):
    if len(incomes) != len(consumptions):
        raise ValueError("Income and consumption lists must have the same length.")

    total = 0
    count = 0
    for i in range(len(incomes)):
        share = saving_share(incomes[i], consumptions[i])
        if share is not None:
            total = total + share
            count = count + 1

    if count == 0:
        return None
    else:
        return total / count


## (c) Four diagnostic input pairs

| Case | Inputs | Expected | Proposed | Defect exposed |
| --- | --- | --- | --- | --- |
| 1 | `[1000, 500]`, `[800, 500]` | `0.1` | `0.2` | Zero share skipped |
| 2 | `[1000, 1000]`, `[800, 1200]` | `0.0` | `-0.1` | Running total reset |
| 3 | `[1000]`, `[800, 500]` | `ValueError` | `0.2` | Missing length check |
| 4 | `[0, 0]`, `[100, 200]` | `None` | `ZeroDivisionError` | Missing empty guard |

Observed calls and results:

```python
>>> proposed_mean([1000, 500], [800, 500])
0.2
```

The shares are 0.2 and 0.0, so the correct mean is 0.1, but the zero share is skipped.

```python
>>> proposed_mean([1000, 1000], [800, 1200])
-0.1
```

The shares are 0.2 and -0.2, so the correct mean is 0.0. The loop resets `total` to the last share (-0.2) but still divides by 2.

```python
>>> proposed_mean([1000], [800, 500])
0.2
```

The second household is silently dropped by `zip`; the contract requires `ValueError`.

```python
>>> proposed_mean([0, 0], [100, 200])
Traceback (most recent call last):
  ...
ZeroDivisionError: division by zero
```

Both shares are `None`, so `count` stays 0 and the division fails; the contract requires `None`.


# 5. Supply evidence and a reproducible hand-in


In [12]:
# (a) Meaningful, passing assertions

import math

# Ordinary saving: 25% of income is saved.
assert math.isclose(saving_share(1000, 750), 0.25)

# Negative saving is valid and keeps its sign.
assert math.isclose(saving_share(1000, 1250), -0.25)

# Zero income gives an undefined share, not zero.
assert saving_share(0, 150) is None

# Eligibility includes exactly EUR 600 per person: 1200 / 2 = 600.
# This catches the mistake of treating exactly EUR 600 per person as ineligible.
assert math.isclose(transfer(1200, 1400, 2), 200)

# Just above the boundary: 1202 / 2 = 601, so no transfer.
assert math.isclose(transfer(1202, 1400, 2), 0)

# Transfer cap: the shortfall is 1000, but the cap is 100 * 3 = 300.
# This catches the mistake of forgetting the EUR 100-per-person cap, or applying it once per household.
assert math.isclose(transfer(0, 1000, 3), 300)

# A household without a shortfall receives zero.
assert math.isclose(transfer(500, 400, 1), 0)

# All shares undefined: the correct answer is None, not 0 or an error.
# This catches the mistake of returning 0 or crashing when every share is undefined.
assert mean_defined_share([0, 0], [100, 200]) is None

# Zero shares are valid and must be included in the mean.
# This catches the mistake of dropping zero shares instead of including them.
assert math.isclose(mean_defined_share([1000, 500], [800, 500]), 0.1)

# The counterfactual did not modify the original income list.
assert income_numbers == original_income_numbers

# Only A01 changed in the counterfactual copy.
assert counterfactual_income_numbers[0] == 1202
assert counterfactual_income_numbers[1:] == income_numbers[1:]

print("All assertions passed.")

All assertions passed.
